# Import libs

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import cv2
import hashlib
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# --- CẤU HÌNH HỆ THỐNG ---
DATA_DIR = "../data/kneeKL224"  # Thay đổi đường dẫn này nếu cần
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ['0', '1', '2', '3', '4']
BATCH_SIZE = 32
EPOCHS = 2  # Chạy thử 2 vòng theo yêu cầu
LEARNING_RATE = 1e-4

print(f"🚀 Thiết bị đang sử dụng: {DEVICE}")

🚀 Thiết bị đang sử dụng: cpu



# Define dataset and Transforms

In [2]:
class KneeDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = os.path.join(root_dir, split)
        self.transform = transform
        self.images = []
        self.labels = []
        
        if not os.path.exists(self.root_dir):
            print(f"⚠️ Cảnh báo: Thư mục {self.root_dir} không tồn tại.")
            return

        for label in CLASSES:
            class_path = os.path.join(self.root_dir, label)
            if os.path.isdir(class_path):
                files = [os.path.join(class_path, f) for f in os.listdir(class_path) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                self.images.extend(files)
                self.labels.extend([int(label)] * len(files))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Augmentation chuẩn cho EfficientNet
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# dataleakage, imbalance, auto-test

In [3]:
def check_data_leakage(train_ds, val_ds, test_ds):
    def get_hashes(dataset):
        hashes = set()
        for img_path in dataset.images:
            with open(img_path, "rb") as f:
                hashes.add(hashlib.md5(f.read()).hexdigest())
        return hashes

    print("\n--- 🔍 Đang kiểm tra Data Leakage ---")
    train_h = get_hashes(train_ds)
    val_h = get_hashes(val_ds)
    test_h = get_hashes(test_ds)
    
    leak_v = train_h.intersection(val_h)
    leak_t = train_h.intersection(test_h)
    
    print(f"Ảnh trùng Train-Val: {len(leak_v)}")
    print(f"Ảnh trùng Train-Test: {len(leak_t)}")
    if len(leak_v) + len(leak_t) > 0:
        print("❌ CẢNH BÁO: Dữ liệu bị rò rỉ!")
    else:
        print("✅ Dữ liệu sạch.")

def get_class_weights(dataset):
    from collections import Counter
    counts = Counter(dataset.labels)
    total = len(dataset.labels)
    # Tính trọng số nghịch đảo để cân bằng nhãn
    weights = [total / (len(CLASSES) * counts[i]) if counts[i] > 0 else 1.0 for i in range(len(CLASSES))]
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)

def verify_auto_test(model):
    print("\n--- 🏁 Kiểm định trên tập AUTO_TEST ---")
    auto_ds = KneeDataset(DATA_DIR, 'auto_test', data_transforms['val'])
    if len(auto_ds) == 0: return
    loader = DataLoader(auto_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print(classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0))

# Build Effecient-Net B0

In [4]:
def build_model():
    print("\n--- 🏗️ Đang khởi tạo EfficientNet-B0 ---")
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, len(CLASSES))
    )
    return model.to(DEVICE)

# Training and Evaluation

In [5]:
# Cập nhật Cell 5 nếu chưa giống thế này:
def train_engine(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs):
    best_acc = 0.0
    for epoch in range(epochs):
        # TRAIN
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        # VAL
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                all_preds.extend(outputs.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_knee_model.pth")
    return all_labels, all_preds

# EXE

In [ ]:
from torch.utils.data import Subset

# --- TÙY CHỈNH CHẠY NHANH ---
FRACTION = 0.1  # Lấy 10% dữ liệu để test code
EPOCHS = 2      # Chạy 2 vòng
# ---------------------------

# 1. Khởi tạo Dataset gốc (Để lấy đường dẫn ảnh)
train_ds_full = KneeDataset(DATA_DIR, 'train', data_transforms['train'])
val_ds_full = KneeDataset(DATA_DIR, 'val', data_transforms['val'])
test_ds_full = KneeDataset(DATA_DIR, 'test', data_transforms['val'])

# 2. Tạo Subset để chạy lẹ
def get_subset(dataset, fraction):
    num_data = len(dataset)
    indices = np.arange(num_data)
    np.random.seed(42) # Cố định để kết quả không đổi mỗi lần chạy
    np.random.shuffle(indices)
    subset_size = int(num_data * fraction)
    return Subset(dataset, indices[:subset_size])

train_subset = get_subset(train_ds_full, FRACTION)
val_subset = get_subset(val_ds_full, FRACTION)
print(f"⚡ Chế độ chạy nhanh: Train ({len(train_subset)} ảnh), Val ({len(val_subset)} ảnh)")

# 3. Kiểm tra dữ liệu & Trọng số (Vẫn dùng ds_full để tính Weight cho chuẩn)
check_data_leakage(train_ds_full, val_ds_full, test_ds_full)
weights = get_class_weights(train_ds_full)

# 4. Khởi tạo DataLoader từ Subset
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

# 5. Khởi tạo Model & Optimizer
model = build_model()
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 6. Huấn luyện
print(f"\n🚀 Đang train thử {EPOCHS} epochs trên {FRACTION*100}% dữ liệu...")
true_labels, pred_labels = train_engine(model, train_loader, val_loader, criterion, optimizer, scheduler, EPOCHS)

# 7. Verify Auto-test (Chạy trên toàn bộ tập auto_test)
verify_auto_test(model)

# 1. Positive Signs (Good signs)

- **Model learning:** Loss decreased from 1.6244 to 1.5224 and Accuracy increased from 26% to 36%. This is a very good signal showing that EfficientNet-B0 is starting to capture meaningful features from the X-ray images.

- **Clean data:** Leakage check returned 0, meaning your results are genuine and not due to "data leakage".

---

# 2. Analysis of Auto-test Results (The Challenges)

From the classification report, there are 3 key issues to note:

- **Grade 0 (Normal) performs best:** Precision 0.46 and Recall 0.58. The model detects healthy cases reasonably well.

- **Grade 4 (Most severe) performs worst:** F1-score only 0.07. This is understandable since the dataset has very few Grade 4 images.

- **Confusion among neighboring classes:** Precision for Grades 1, 2, and 3 is quite low (0.19 - 0.32). This is typical for osteoarthritis because the boundary between Grade 1 and Grade 2 is very subtle (mild joint space narrowing vs small osteophytes).

---

# 3. Proposed Adjustments to Achieve Higher Accuracy (>70-80%)

When you move to run on full data (FRACTION = 1.0), consider the following steps:

### A. Increase Number of Epochs

- 2 epochs is too few for the model to converge.

- Recommended: Set **EPOCHS = 20 to 30**.

- Use **Early Stopping** (stop if validation loss does not improve after 5 epochs) to avoid overfitting.

### B. Adjust Learning Rate

- Current learning rate of 1e-4 (0.0001) is a bit conservative.

- Try using optimizer:  
  ```python
  optimizer = optim.AdamW(model.parameters(), lr=3e-4)
  ```

- Replace StepLR with **ReduceLROnPlateau** scheduler to automatically reduce LR when loss plateaus.

### C. Handle Class Imbalance More Deeply

- Although WeightedLoss is used, Grade 4 is very rare.

- Try **oversampling**: augment Grade 4 images more aggressively (rotation, flipping, brightness adjustment) specifically in your `KneeDataset`.

### D. Check Image Size

- Knee X-rays require high detail.

- If 224x224 resolution is insufficient, consider upgrading to EfficientNet-B3 with input size 300x300 or higher for better performance.

---
```


(Bổ sung Sampler vào phần xử lý Imbalance)

In [6]:
from torch.utils.data import WeightedRandomSampler
from collections import Counter

def get_imbalance_tools(dataset):
    # 1. Tính toán trọng số cho từng lớp
    counts = Counter(dataset.labels)
    class_weights = [len(dataset) / counts[i] for i in range(len(CLASSES))]
    
    # 2. Tạo Sampler (Quan trọng: Giúp Grade 4 xuất hiện nhiều hơn trong tập Train)
    sample_weights = [class_weights[label] for label in dataset.labels]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    
    # 3. Trọng số cho Loss function (Phạt nặng hơn khi sai lớp ít dữ liệu)
    loss_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
    
    return sampler, loss_weights

# Lưu ý: Sampler chỉ dùng cho tập TRAIN

In [7]:
def build_model():
    print("\n--- 🏗️ Đang khởi tạo EfficientNet-B0 (CPU Optimized) ---")
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    
    # FREEZE: Không tính toán gradient cho các lớp feature để CPU chạy nhanh hơn
    for param in model.parameters():
        param.requires_grad = False
    
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, len(CLASSES))
    )
    # Chỉ train lớp classifier cuối cùng
    return model.to(DEVICE)

In [11]:
# --- CẤU HÌNH ---
FRACTION = 0.02  # Bạn có thể để 0.1 nếu muốn test nhanh lần nữa
EPOCHS = 1
BATCH_SIZE = 8

# ----------------

# 1. Load Data
train_ds = KneeDataset(DATA_DIR, 'train', data_transforms['train'])
val_ds = KneeDataset(DATA_DIR, 'val', data_transforms['val'])

# 2. Lấy công cụ xử lý Imbalance (Sampler & Weights)
train_sampler, loss_weights = get_imbalance_tools(train_ds)

# 3. Khởi tạo DataLoader
# Lưu ý quan trọng: Khi dùng sampler, shuffle PHẢI bằng False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 4. Model & Loss (Label Smoothing giúp giảm overfitting)
model = build_model()
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.1)

# 5. Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# ĐÃ SỬA: Xóa tham số verbose=True để tránh lỗi TypeError
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.1, 
    patience=3
)

# 6. Hàm huấn luyện nâng cao
def train_engine_v2(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs):
    best_acc = 0.0
    for epoch in range(epochs):
        # --- TRAIN PHASE ---
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        
        # --- VAL PHASE ---
        model.eval()
        val_loss, val_correct = 0.0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                all_preds.extend(outputs.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Tính toán chỉ số trung bình
        avg_train_loss = train_loss / len(train_loader.dataset)
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        
        # Cập nhật Scheduler dựa trên Validation Loss
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        
        # In thông báo thay đổi LR nếu có
        lr_msg = f" | LR: {new_lr:.1e}" if new_lr < old_lr else ""
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}{lr_msg}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_knee_model.pth")
            print(f"🌟 Đã lưu model mới với Acc: {val_acc:.4f}")
            
    return all_labels, all_preds

# Chạy huấn luyện
print(f"🚀 Bắt đầu huấn luyện với EfficientNet-B0...")
true_labels, pred_labels = train_engine_v2(model, train_loader, val_loader, criterion, optimizer, scheduler, EPOCHS)

# Verify Auto-test cuối cùng
verify_auto_test(model)


--- 🏗️ Đang khởi tạo EfficientNet-B0 (CPU Optimized) ---
🚀 Bắt đầu huấn luyện với EfficientNet-B0...
Epoch 1/1 | Train Loss: 1.3376 | Val Loss: 2.1432 | Val Acc: 0.0460
🌟 Đã lưu model mới với Acc: 0.0460

--- 🏁 Kiểm định trên tập AUTO_TEST ---
              precision    recall  f1-score   support

           0       0.71      0.01      0.02       604
           1       0.25      0.08      0.13       275
           2       0.00      0.00      0.00       403
           3       0.11      0.01      0.02       200
           4       0.03      1.00      0.06        44

    accuracy                           0.05      1526
   macro avg       0.22      0.22      0.04      1526
weighted avg       0.34      0.05      0.03      1526

